# Survival Analysis Agent - Cancer Research Example

This notebook demonstrates how to use the **SurvivalAnalysisAgent** for analyzing time-to-event data in cancer research.

The agent can perform:
- Kaplan-Meier survival curves
- Cox proportional hazards modeling  
- Risk stratification
- Group comparisons (log-rank test)
- Survival predictions

## Use Cases
- Patient survival analysis
- Treatment efficacy comparison
- Prognostic factor identification
- Clinical trial outcomes
- Risk group stratification

## Setup

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import os

from langchain_openai import ChatOpenAI
from ai_data_science_team.ml_agents import SurvivalAnalysisAgent

In [ ]:
# Set your OpenAI API key
os.environ['OPENAI_API_KEY'] = "your-api-key-here"

# Initialize language model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm

## Cost-Effective Alternative: OpenRouter

**Save 10-100x on API costs** by using OpenRouter instead of direct OpenAI API!

OpenRouter provides access to multiple LLM providers at significantly lower costs:
- **Claude 3.5 Sonnet**: ~$3/M tokens (vs $15 direct)
- **Claude 3 Haiku**: ~$0.25/M tokens (vs $1 direct)  
- **Llama 3.1 70B**: ~$0.35/M tokens (open source)
- **Gemini Pro 1.5**: ~$1.25/M tokens

### Setup OpenRouter

1. Get API key from https://openrouter.ai/
2. Set environment variable
3. Use `get_openrouter_llm()` helper function

In [ ]:
# Option 1: Use OpenRouter with Claude 3.5 Sonnet (high quality, ~$3/M tokens)
from ai_data_science_team.utils.openrouter import get_openrouter_llm

# Set your OpenRouter API key
os.environ['OPENROUTER_API_KEY'] = "sk-or-v1-..."

# Create OpenRouter LLM (drop-in replacement for ChatOpenAI)
llm_openrouter = get_openrouter_llm(
    model="anthropic/claude-3.5-sonnet",
    temperature=0
)

llm_openrouter

In [ ]:
# Option 2: Budget-friendly models for testing/development
from ai_data_science_team.utils.openrouter import compare_costs, list_recommended_models

# See all recommended models
models = list_recommended_models()
print("Premium models:")
for model, info in models['premium'].items():
    print(f"  {model}: {info['price']} - {info['use_case']}")

print("\nBalanced models:")
for model, info in models['balanced'].items():
    print(f"  {model}: {info['price']} - {info['use_case']}")

print("\nBudget models:")
for model, info in models['budget'].items():
    print(f"  {model}: {info['price']} - {info['use_case']}")

# Compare costs (example: 1M tokens)
print("\n" + "="*60)
compare_costs(
    openai_model="gpt-4o",
    openrouter_model="anthropic/claude-3.5-sonnet",
    tokens=1_000_000
)

In [ ]:
# Option 3: Use with SurvivalAnalysisAgent (example)
# Just replace the 'model' parameter with your OpenRouter LLM

# For production cancer research (high quality):
llm_production = get_openrouter_llm("anthropic/claude-3.5-sonnet")

# For development/testing (budget-friendly):
llm_budget = get_openrouter_llm("anthropic/claude-3-haiku")

# For high-volume batch processing (cheapest):
llm_batch = get_openrouter_llm("meta-llama/llama-3.1-70b-instruct")

# Use with agent - exactly the same interface!
# survival_agent = SurvivalAnalysisAgent(
#     model=llm_production,  # or llm_budget, llm_batch
#     time_column="survival_months",
#     event_column="death_event"
# )

In [ ]:
# Option 4: Ultra-Budget Chinese Models (100-500x cheaper than OpenAI!)
# Perfect for massive batch processing or budget-constrained research

# DeepSeek Chat - ULTRA CHEAP, good for code generation
llm_deepseek = get_openrouter_llm("deepseek/deepseek-chat")

# Qwen 2.5 72B - Alibaba's flagship model, multilingual support
llm_qwen = get_openrouter_llm("qwen/qwen-2.5-72b-instruct")

# Yi Large - Strong reasoning at ultra-low cost
llm_yi = get_openrouter_llm("01-ai/yi-large")

# GLM-4 9B - ChatGLM model, extremely cheap for simple tasks
llm_glm = get_openrouter_llm("zhipu/glm-4-9b-chat")

# Example cost comparison for 50,000 patient analysis:
# - OpenAI GPT-4o: ~$150
# - DeepSeek Chat: ~$0.50 (99.7% cheaper!)
# - Qwen 2.5 72B: ~$1.25 (99.2% cheaper!)

print("Ultra-budget models loaded!")
print("Use these for massive batch processing and save hundreds of dollars!")

## Generate Synthetic Cancer Survival Data

We'll create synthetic data mimicking a cancer clinical trial with:
- Survival time (months)
- Death event indicator
- Treatment group (A vs B)
- Age, tumor stage, biomarker levels

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Number of patients
n_patients = 200

# Generate synthetic cancer patient data
data = pd.DataFrame({
    'patient_id': [f'P{i:04d}' for i in range(1, n_patients + 1)],
    'age': np.random.normal(65, 10, n_patients).clip(40, 90).astype(int),
    'treatment_group': np.random.choice(['Treatment_A', 'Treatment_B'], n_patients),
    'tumor_stage': np.random.choice(['Stage_I', 'Stage_II', 'Stage_III', 'Stage_IV'], 
                                     n_patients, p=[0.2, 0.3, 0.3, 0.2]),
    'biomarker_level': np.random.lognormal(0, 0.5, n_patients).round(2),
    'gender': np.random.choice(['Male', 'Female'], n_patients),
})

# Generate survival times (influenced by treatment and stage)
base_survival = 24  # months

# Treatment B is more effective (longer survival)
treatment_effect = np.where(data['treatment_group'] == 'Treatment_B', 12, 0)

# Advanced stages have worse survival
stage_effect = data['tumor_stage'].map({
    'Stage_I': 18,
    'Stage_II': 6,
    'Stage_III': -6,
    'Stage_IV': -18
})

# Age effect (older = slightly worse)
age_effect = -(data['age'] - 65) * 0.3

# Generate survival times with some randomness
data['survival_months'] = (
    base_survival + 
    treatment_effect + 
    stage_effect + 
    age_effect +
    np.random.exponential(8, n_patients)
).clip(0.5, None).round(1)

# Generate event indicator (death = 1, censored = 0)
# About 30% censoring rate
data['death_event'] = np.where(
    np.random.random(n_patients) < 0.7,
    1,  # Death occurred
    0   # Censored (patient still alive at end of study)
)

# For censored patients, survival_months represents follow-up time
print(f"Dataset created with {len(data)} patients")
print(f"Events (deaths): {data['death_event'].sum()} ({data['death_event'].mean()*100:.1f}%)")
print(f"Censored: {(1-data['death_event']).sum()} ({(1-data['death_event'].mean())*100:.1f}%)")

data.head(10)

In [ ]:
# View summary statistics
data.describe()

## Example 1: Kaplan-Meier Survival Curves by Treatment Group

In [ ]:
# Initialize Survival Analysis Agent
survival_agent = SurvivalAnalysisAgent(
    model=llm,
    time_column="survival_months",
    event_column="death_event",
    log=True,
    log_path="logs/",
    n_samples=20
)

survival_agent

In [ ]:
# Perform Kaplan-Meier analysis comparing treatment groups
survival_agent.invoke_agent(
    data_raw=data,
    user_instructions="""
    Create Kaplan-Meier survival curves stratified by treatment_group.
    Compare Treatment_A vs Treatment_B.
    Include confidence intervals and perform log-rank test.
    Return an interactive Plotly visualization.
    """,
    max_retries=3,
    retry_count=0
)

In [ ]:
# Get survival results
survival_results = survival_agent.get_survival_results()
survival_results

In [ ]:
# Display the survival curve
fig = survival_agent.get_plotly_graph()
if fig:
    fig.show()

In [ ]:
# View the generated survival analysis code
survival_agent.get_survival_analyzer_function(markdown=True)

## Example 2: Cox Proportional Hazards Model

In [ ]:
# Create a new agent for Cox regression
cox_agent = SurvivalAnalysisAgent(
    model=llm,
    time_column="survival_months",
    event_column="death_event",
    log=True,
    log_path="logs/"
)

# Fit Cox proportional hazards model
cox_agent.invoke_agent(
    data_raw=data,
    user_instructions="""
    Fit a Cox proportional hazards model to identify prognostic factors.
    Include these covariates: treatment_group, tumor_stage, age, biomarker_level.
    Report hazard ratios with 95% confidence intervals and p-values.
    Create a forest plot showing hazard ratios.
    """,
    max_retries=3
)

In [ ]:
# Get Cox model results
cox_results = cox_agent.get_survival_results()
cox_results

In [ ]:
# View forest plot if generated
cox_fig = cox_agent.get_plotly_graph()
if cox_fig:
    cox_fig.show()

## Example 3: Risk Stratification

In [ ]:
# Create agent for risk stratification
risk_agent = SurvivalAnalysisAgent(
    model=llm,
    time_column="survival_months",
    event_column="death_event",
    log=True
)

# Stratify patients into risk groups
risk_agent.invoke_agent(
    data_raw=data,
    user_instructions="""
    Perform risk stratification using a Cox model.
    Divide patients into Low, Medium, and High risk groups based on predicted risk scores.
    Create Kaplan-Meier curves for each risk group.
    Report median survival for each risk group.
    """,
    max_retries=3
)

In [ ]:
# Get risk stratification results
risk_results = risk_agent.get_survival_results()
risk_results

In [ ]:
# View risk group survival curves
risk_fig = risk_agent.get_plotly_graph()
if risk_fig:
    risk_fig.show()

## Example 4: Subgroup Analysis by Tumor Stage

In [ ]:
# Create agent for stage-specific analysis
stage_agent = SurvivalAnalysisAgent(
    model=llm,
    time_column="survival_months",
    event_column="death_event"
)

# Analyze survival by tumor stage
stage_agent.invoke_agent(
    data_raw=data,
    user_instructions="""
    Compare survival across tumor stages (Stage_I, Stage_II, Stage_III, Stage_IV).
    Create Kaplan-Meier curves for each stage.
    Perform pairwise log-rank tests.
    Report median survival times for each stage.
    """
)

In [ ]:
# View stage-specific results
stage_results = stage_agent.get_survival_results()
stage_results

In [ ]:
# Display survival curves by stage
stage_fig = stage_agent.get_plotly_graph()
if stage_fig:
    stage_fig.show()

## Example 5: Survival Prediction for New Patients

In [ ]:
# Create new patient data for prediction
new_patients = pd.DataFrame({
    'patient_id': ['NEW001', 'NEW002', 'NEW003'],
    'age': [55, 70, 62],
    'treatment_group': ['Treatment_B', 'Treatment_A', 'Treatment_B'],
    'tumor_stage': ['Stage_II', 'Stage_III', 'Stage_I'],
    'biomarker_level': [1.2, 2.5, 0.8],
    'gender': ['Female', 'Male', 'Male'],
    'survival_months': [0, 0, 0],  # To be predicted
    'death_event': [0, 0, 0]  # Unknown
})

new_patients

In [ ]:
# Combine training and new patient data
combined_data = pd.concat([data, new_patients], ignore_index=True)

# Create prediction agent
pred_agent = SurvivalAnalysisAgent(
    model=llm,
    time_column="survival_months",
    event_column="death_event"
)

# Make survival predictions
pred_agent.invoke_agent(
    data_raw=combined_data,
    user_instructions="""
    Train a Cox model on patients with survival data.
    Predict survival probabilities at 12, 24, and 36 months for the 3 new patients (NEW001-NEW003).
    Calculate their risk scores and expected survival times.
    """
)

In [ ]:
# Get prediction results
predictions = pred_agent.get_survival_results()
predictions

## Summary

The **SurvivalAnalysisAgent** provides:

✅ **Automated survival analysis** - No manual coding required  
✅ **Clinical research ready** - Kaplan-Meier, Cox regression, risk stratification  
✅ **Interactive visualizations** - Plotly charts for presentations  
✅ **Reproducible** - All code is logged and can be reused  
✅ **Flexible** - Natural language instructions for any analysis  

## Next Steps

1. **Real cancer datasets**: Use with TCGA, SEER, or clinical trial data
2. **Multi-agent workflows**: Combine with DataCleaningAgent, FeatureEngineeringAgent
3. **Custom analyses**: Time-dependent covariates, competing risks
4. **Integration**: Connect with cBioPortal via pyBioPortal
5. **Reporting**: Generate publication-ready figures and tables